In [1]:
!pip install transformers torch -q
import torch
import pandas as pd
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
model_path = '/content/drive/MyDrive/PolyGuard/03-models/polyguard_model'

tokenizer = AutoTokenizer.from_pretrained(model_path)
model = AutoModelForSequenceClassification.from_pretrained(model_path)
model.eval()

print("Model loaded!")

Loading weights:   0%|          | 0/201 [00:01<?, ?it/s]

Model loaded!


In [3]:
def get_score(code_text):
    # tokenize the input code
    inputs = tokenizer(
        code_text,
        return_tensors="pt",
        truncation=True,
        max_length=256,
        padding=True
    )

    # get model prediction
    with torch.no_grad():
        outputs = model(**inputs)

    # convert to probabilities
    probs = torch.softmax(outputs.logits, dim=1)
    clean_confidence = probs[0][0].item()      # probability it is clean
    vuln_confidence  = probs[0][1].item()      # probability it is vulnerable

    # convert to 0-10 score
    score = round(clean_confidence * 10, 1)

    return {
        "score": score,
        "clean_confidence": round(clean_confidence * 100, 1),
        "vuln_confidence":  round(vuln_confidence  * 100, 1),
        "verdict": "CLEAN" if score >= 7 else "VULNERABLE"
    }

print("Score function ready!")

Score function ready!


In [4]:
suggestions = {
    "sqli":    "Use parameterized queries instead of building SQL strings manually.",
    "xss":     "Sanitize all user inputs before rendering them to the page.",
    "secrets": "Never hardcode API keys or passwords. Use environment variables instead.",
    "crypto":  "Avoid MD5 and SHA1. Use SHA256 or bcrypt for hashing.",
    "memory":  "Always check buffer sizes before copying data in C/C++.",
    "auth":    "Always verify user permissions before returning sensitive data.",
}

def get_suggestion(vuln_type):
    return suggestions.get(vuln_type, "Review this code carefully for security issues.")

print("Suggestions ready!")

Suggestions ready!


In [5]:
language_tips = {
    "python": [
        "Use list comprehensions instead of for loops where possible.",
        "Use 'with open()' instead of open/close for file handling.",
        "Use f-strings for string formatting instead of .format() or %.",
    ],
    "javascript": [
        "Use 'const' and 'let' instead of 'var'.",
        "Use async/await instead of nested callbacks.",
        "Always use === instead of == for comparisons.",
    ],
    "java": [
        "Use try-with-resources for handling streams and connections.",
        "Prefer interfaces over abstract classes when possible.",
        "Use StringBuilder instead of String concatenation in loops.",
    ],
    "go": [
        "Always handle errors explicitly — never ignore the error return value.",
        "Use goroutines for concurrency instead of threads.",
        "Keep functions small and focused on one task.",
    ],
}

def get_language_tips(language):
    tips = language_tips.get(language.lower(), ["No tips available for this language yet."])
    return tips

print("Language tips ready!")

Language tips ready!


In [6]:
# test with a vulnerable code example
sample_code = """
import sqlite3
user_input = "1 OR 1=1"
query = "SELECT * FROM users WHERE id = " + user_input
conn = sqlite3.connect('db.sqlite')
conn.execute(query)
"""

result = get_score(sample_code)
print("=== POLYGUARD RESULT ===")
print(f"Score:   {result['score']} / 10")
print(f"Verdict: {result['verdict']}")
print(f"Clean confidence:  {result['clean_confidence']}%")
print(f"Vuln confidence:   {result['vuln_confidence']}%")
print()
print("Suggestion:", get_suggestion("sqli"))
print()
print("Python tips:")
for tip in get_language_tips("python"):
    print(" -", tip)

=== POLYGUARD RESULT ===
Score:   5.1 / 10
Verdict: VULNERABLE
Clean confidence:  50.9%
Vuln confidence:   49.1%

Suggestion: Use parameterized queries instead of building SQL strings manually.

Python tips:
 - Use list comprehensions instead of for loops where possible.
 - Use 'with open()' instead of open/close for file handling.
 - Use f-strings for string formatting instead of .format() or %.
